# ⚡ ML STA ENGINE — Physics-Informed ML Framework
### Inspired by: *Delay Models for CMOS Circuits* · McFarland & Flynn · Stanford 1995 · CSL-TR-95-672

---

**Loads your local `CMOS_Delay_Dataset.csv`** — no data generation required.

| Section | Content |
|---|---|
| 0 | Setup, imports, config |
| 1 | Dataset loading & column adapter |
| 2 | Data exploration |
| 3 | Physics feature engineering (4 analytical models) |
| 4 | Data pipeline (PyTorch Dataset + DataLoaders) |
| 5A | PhysicsResidualNet — PINN (residual over analytical baseline) |
| 5B | FT-Transformer — tabular attention model |
| 6 | Physics-Informed Loss (monotonicity constraints via autograd) |
| 7 | AMP + DataParallel training (2× T4 GPU) |
| 8 | Baselines: XGBoost · LightGBM · Random Forest |
| 9 | Evaluation vs all 4 Stanford analytical models |
| 10 | Visualization (10 publication-quality plots) |
| 11 | Explainability: SHAP |
| 12 | Run everything — `main()` |
| 13 | Flask REST API |
| 14 | Streamlit dashboard |

In [1]:
# Run once — Kaggle already has torch/sklearn/xgboost; just add extras
!pip install lightgbm shap flask flask-cors streamlit plotly -q

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPUs     : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i} : {p.name}  {p.total_memory//1024**3} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 58.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 104.5 MB/s eta 0:00:0000:010:01
PyTorch  : 2.10.0+cu128
CUDA     : True
GPUs     : 2
  GPU 0 : Tesla T4  14 GB
  GPU 1 : Tesla T4  14 GB


## Section 0 — Imports & Configuration

In [2]:
import os, sys, math, time, json, pickle, warnings, logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor

try:   import xgboost  as xgb; HAS_XGB  = True
except ImportError:              HAS_XGB  = False

try:   import lightgbm as lgb; HAS_LGB  = True
except ImportError:              HAS_LGB  = False

try:   import shap;            HAS_SHAP = True
except ImportError:              HAS_SHAP = False

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger(__name__)

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

PALETTE = {
    "PINN": "#E63946", "FT-Transformer": "#457B9D",
    "XGBoost": "#2A9D8F", "LightGBM": "#E9C46A", "RandomForest": "#F4A261",
    "1-Region": "#6B705C", "2-Region": "#A5A58D",
    "3-Region": "#B7B7A4", "Alpha-Power": "#C9C9B5",
}

# ── Series-transistor table (McFarland & Flynn, Section 10) ─────────────────
_SERIES_N = {1:(11.51,0.313), 2:(9.13,0.372), 3:(8.26,0.404), 4:(7.78,0.430)}
_SERIES_P = {1:(27.41,0.308), 2:(21.70,0.414), 3:(19.22,0.484), 4:(19.69,0.488)}
_CORNER   = {
    "TT":(1.00,1.00,1.00,1.00), "FF":(0.75,0.90,0.75,0.90),
    "SS":(1.35,1.10,1.35,1.10), "FS":(0.75,0.90,1.35,1.10),
    "SF":(1.35,1.10,0.75,0.90),
}

In [3]:
# Replace your Config class in Section 0 with this:
@dataclass
class Config:
    OUT_DIR:  str  = "cmos_output"
    SEED:     int  = 42
    TEST_FRAC:float= 0.15
    VAL_FRAC: float= 0.10
    # Paper calibrated constants
    RF_N: float = 5.31;   RF_P: float = 14.24   # kΩ·μm
    ALPHA: float = 1.4
    
    # Model dims
    HIDDEN: list = field(default_factory=lambda: [512, 256, 128, 64])
    DROP:   float= 0.10  
    FT_D:   int  = 64;  FT_H: int = 8;  FT_L: int = 4;  FT_DR: float = 0.10
    
    # Training
    # FIX: Dropped Batch Size from 2048 to 256. This gives the optimizer
    # 8x more gradient updates per epoch to fine-tune tiny delay values!
    BATCH:   int  = 256  
    EPOCHS:  int  = 150  
    LR:      float= 1e-3;  WD: float = 1e-4  # Increased LR slightly for smaller batch
    CLIP:    float= 1.0;   PATIENCE: int = 25
    
# Replace this part inside your Config class in Section 0
    # FIX: Physics Regularization
    LAM_PHY: float= 0.0  # LAM_PHY=0.0: physics off (constraints hurt due to sampling bias - see paper Sec. 4)
    LAM_SM:  float= 0.0
    
    # Baselines
    RF_N_EST:int  = 300
    XGB_N:   int  = 1000;  XGB_D: int = 8;  XGB_LR: float = 0.05
    LGB_N:   int  = 1000;  LGB_D: int = 8;  LGB_LR: float = 0.05

    def __post_init__(self):
        for sub in ("", "/models", "/plots", "/results"):
            os.makedirs(self.OUT_DIR + sub, exist_ok=True)
        torch.manual_seed(self.SEED); np.random.seed(self.SEED)
        self.DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.N_GPUS  = torch.cuda.device_count()
        self.USE_DP  = self.N_GPUS > 1
        self.USE_AMP = torch.cuda.is_available()
        log.info(f"Device={self.DEVICE}  GPUs={self.N_GPUS}  AMP={self.USE_AMP}")

CFG = Config()
TARGETS = ["rise_delay_ns", "fall_delay_ns", "avg_delay_ns"]
print("Config ready  ✓")

2026-06-14 06:20:57,603 [INFO] Device=cuda  GPUs=2  AMP=True


Config ready  ✓


## Section 1 — Dataset Loading

> **Specify the path to your `CMOS_Delay_Dataset.csv`** in the cell below.
>
> | Platform | Example path |
> |---|---|
> | Kaggle (upload as dataset) | `/kaggle/input/cmos-delay/CMOS_Delay_Dataset.csv` |
> | Kaggle (notebook file upload) | `CMOS_Delay_Dataset.csv` |
> | Colab (Drive mount) | `/content/drive/MyDrive/CMOS_Delay_Dataset.csv` |
> | Local / Windows | `C:/Users/YourName/Downloads/CMOS_Delay_Dataset.csv` |
> | Local / Linux-Mac | `~/Downloads/CMOS_Delay_Dataset.csv` |

In [4]:
# ╔══════════════════════════════════════════════════════╗
# ║   ← CHANGE THIS PATH TO YOUR CSV FILE LOCATION      ║
DATASET_PATH = "/kaggle/input/datasets/rishik25t4/cmos-dataset123/Modern_45nm_Delay_Dataset_cleaned.csv"
# ╚══════════════════════════════════════════════════════╝

df_raw = pd.read_csv(DATASET_PATH)
print(f"Loaded  : {DATASET_PATH}")
print(f"Shape   : {df_raw.shape[0]:,} rows  ×  {df_raw.shape[1]} columns")
print(f"Columns : {list(df_raw.columns)}")
print(f"Missing : {df_raw.isnull().sum().sum()}")
print("\nFirst 3 rows:")
display(df_raw.head(3))

Loaded  : /kaggle/input/datasets/rishik25t4/cmos-dataset123/Modern_45nm_Delay_Dataset_cleaned.csv
Shape   : 11,322 rows  ×  14 columns
Columns : ['Gate_Type', 'NMOS_Width_nm', 'PMOS_Width_nm', 'Channel_Length_nm', 'VDD_V', 'Input_Slew_ps', 'Load_Capacitance_fF', 'Temperature_C', 'Threshold_Voltage_N_V', 'Threshold_Voltage_P_V', 'Process_Corner', 'rise_delay_ns', 'fall_delay_ns', 'average_delay_ns']
Missing : 28

First 3 rows:


,Gate_Type,NMOS_Width_nm,PMOS_Width_nm,Channel_Length_nm,VDD_V,Input_Slew_ps,Load_Capacitance_fF,Temperature_C,Threshold_Voltage_N_V,Threshold_Voltage_P_V,Process_Corner,rise_delay_ns,fall_delay_ns,average_delay_ns
0,INV,90.0,135.0,45.0,0.8,10.0,1.0,0.0,0.32,0.3,TT,0.003111,0.004744,0.003928
1,INV,90.0,135.0,45.0,0.8,10.0,5.0,0.0,0.32,0.3,TT,0.008468,0.012238,0.010353
2,INV,90.0,135.0,45.0,0.8,50.0,5.0,0.0,0.32,0.3,TT,0.009305,0.017269,0.013287


### Column Adapter
Maps the CSV column names / units → model-standard format and creates the three
prediction targets (`rise_delay_ns`, `fall_delay_ns`, `avg_delay_ns`).

**Rise/fall rationale (physics-based):**

| Gate | Pull-down path | Pull-up path | Critical delay |
|---|---|---|---|
| INV | 1 NMOS | 1 PMOS | balanced  (rise ≈ fall) |
| NAND2 | 2 NMOS **series** (↑R) | 2 PMOS parallel (↓R) | **fall** is slower |
| NOR2  | 2 NMOS parallel (↓R) | 2 PMOS **series** + low μ | **rise** is slower |

In [5]:
# ── Mappings (Make sure these are included!) ─────────────────────────────
CORNER_MAP = {"TT": 0, "FF": 1, "SS": 2, "FS": 3, "SF": 4}
GATE_MAP   = {"INV": 0, "NAND2": 1, "NOR2": 2}

In [6]:
# ── Mappings ─────────────────────────────
NSERIES = {"INV": 1, "NAND2": 2, "NOR2": 2}

def adapt_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Convert corrected CMOS CSV → model-standard DataFrame."""
    out = pd.DataFrame()

    # ── Unit conversions ────────────────────────────────────────────────────
    out["Wn_um"]       = df["NMOS_Width_nm"]     / 1000.0
    out["Wp_um"]       = df["PMOS_Width_nm"]     / 1000.0
    out["L_um"]        = df["Channel_Length_nm"] / 1000.0
    out["TIN_ns"]      = df["Input_Slew_ps"]     / 1000.0

    # ── Direct copies ────────────────────────────────────────────────────────
    out["CLOAD_fF"]    = df["Load_Capacitance_fF"]
    out["VDD_V"]       = df["VDD_V"]
    out["Temperature"] = df["Temperature_C"]
    out["Vth_N_V"]     = df["Threshold_Voltage_N_V"]
    out["Vth_P_V"]     = df["Threshold_Voltage_P_V"]

    # ── Defaults ─────────────────────────────────────────────────────────────
    out["CW_fF"]       = 0.0    
    out["RW_ohm"]      = 0.0    
    out["Fanout"]      = 4.0    

    # ── Categorical variables ────────────────────────────────────────────────
    out["N_series"]       = df["Gate_Type"].map(NSERIES).fillna(1).astype(float)
    out["Gate_Type"]      = df["Gate_Type"]
    out["Process_Corner"] = df["Process_Corner"]

    # ── TARGETS (Using the corrected Average_ns column) ─────────────────────
    out["rise_delay_ns"]  = df["rise_delay_ns"]
    out["fall_delay_ns"]  = df["fall_delay_ns"]
    # Changed from Delay_ps to Average_ns below:
    out["avg_delay_ns"]   = df["average_delay_ns"] 

    return out.reset_index(drop=True)

# Apply and prepare for training
df = adapt_dataset(df_raw)

# One-hot encoding for the models
gate_text = df['Gate_Type']
corner_text = df['Process_Corner']
df = pd.get_dummies(df, columns=['Gate_Type', 'Process_Corner'], dtype=float)
df['Gate_Type'] = gate_text
df['Process_Corner'] = corner_text

print(f"Adapted Dataset Shape: {df.shape} ✓")

Adapted Dataset Shape: (11322, 26) ✓


## Section 2 — Data Exploration

In [7]:
# ## Section 2 — Data Exploration
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0,0].hist(df["avg_delay_ns"]*1000, bins=60, color="#2A9D8F", edgecolor="white")
axes[0,0].set_xlabel("Avg Delay (ps)"); axes[0,0].set_title("Delay Distribution")

df.boxplot(column="avg_delay_ns", by="Process_Corner", ax=axes[0,1])
axes[0,1].set_title("Delay by Process Corner"); axes[0,1].set_xlabel("Corner")

g_mean = df.groupby("Gate_Type")["avg_delay_ns"].mean()
g_mean.plot(kind="bar", ax=axes[0,2], color=["#E63946"])
axes[0,2].set_title("Mean Delay by Gate Type"); axes[0,2].set_xlabel("")

axes[1,0].scatter(df["TIN_ns"]*1000, df["avg_delay_ns"]*1000, alpha=0.15, s=5)
axes[1,0].set_xlabel("Input Slew (ps)"); axes[1,0].set_ylabel("Avg Delay (ps)")
axes[1,0].set_title("Delay vs Input Slew")

axes[1,1].scatter(df["CLOAD_fF"], df["avg_delay_ns"]*1000, alpha=0.15, s=5, color="#E9C46A")
axes[1,1].set_xlabel("CLOAD (fF)"); axes[1,1].set_ylabel("Avg Delay (ps)")
axes[1,1].set_title("Delay vs CLOAD")

axes[1,2].scatter(df["Temperature"], df["avg_delay_ns"]*1000, alpha=0.15, s=5, color="#F4A261")
axes[1,2].set_xlabel("Temperature (C)"); axes[1,2].set_ylabel("Avg Delay (ps)")
axes[1,2].set_title("Delay vs Temperature")

fig.suptitle("Modern 45nm CMOS Delay Dataset — Exploratory Analysis", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f"{CFG.OUT_DIR}/plots/00_eda.png", bbox_inches="tight", dpi=110)
plt.show()

## Section 3 — Physics Feature Engineering

Computes all **4 Stanford analytical delay models** as engineered features fed to the NN.
Technology scaling: $R_M(L) = R_{M,ref} \cdot (L_{\mu m} / 0.8)$ so delay time constants
stay in the correct picosecond range for nanoscale nodes.

In [8]:
class PhysicsExtractor:
    def __init__(self, alpha: float = 1.4):
        self.alpha = alpha

    @staticmethod
    def one_region(TF, dV=0.5): return TF * np.log(1.0 / (1.0 - dV))

    @staticmethod
    def two_region(TIN, TF, TM, VT, dV=0.5):
        tv  = TIN * VT
        sq  = np.sqrt(np.maximum((TF * np.log(1.0 - dV))**2 + 2*TIN*TM*dV, 0))
        return tv + sq

    @staticmethod
    def three_region(TIN, TM, VT, dV=0.5):
        return TIN*(1+VT)/2 + TM*dV / np.maximum(1-VT, 1e-6)

    def alpha_power(self, TIN, TM, VT, dV=0.5):
        a = self.alpha
        return TIN*(a+VT)/(1+a) + TM*dV / np.maximum(1-VT, 1e-6)

    def extract(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        Wn, Wp = df["Wn_um"].values, df["Wp_um"].values
        CL, TIN, L = df["CLOAD_fF"].values, df["TIN_ns"].values, df["L_um"].values
        NS = df["N_series"].values.clip(1, 4).astype(int)

        L_scale = L / 0.8  
        RM_N = np.array([_SERIES_N[n][0] for n in NS]) * L_scale
        VT_N = np.array([_SERIES_N[n][1] for n in NS])
        RM_P = np.array([_SERIES_P[n][0] for n in NS]) * L_scale
        VT_P = np.array([_SERIES_P[n][1] for n in NS])

        TM_N = RM_N * CL / Wn / 1000.0
        TF_N = CFG.RF_N * L_scale * CL / Wn / 1000.0
        TM_P = RM_P * CL / Wp / 1000.0
        TF_P = CFG.RF_P * L_scale * CL / Wp / 1000.0

        out["TM_N_ns"], out["TF_N_ns"] = TM_N, TF_N
        out["TM_P_ns"], out["TF_P_ns"] = TM_P, TF_P

        for tag, TM_a, TF_a, VT_a in [("N", TM_N, TF_N, VT_N), ("P", TM_P, TF_P, VT_P)]:
            out[f"d1R_{tag}"] = self.one_region(TF_a)
            out[f"d2R_{tag}"] = self.two_region(TIN, TF_a, TM_a, VT_a)
            out[f"d3R_{tag}"] = self.three_region(TIN, TM_a, VT_a)
            out[f"dAP_{tag}"] = self.alpha_power(TIN, TM_a, VT_a)

        out["CL_over_W"]   = CL / np.maximum(Wn, 1e-4)
        out["TIN_over_TM"] = TIN / np.maximum(TM_N, 1e-9)
        out["Wn_Wp_ratio"] = Wn / np.maximum(Wp, 1e-4)
        out["log_FO"]      = np.log1p(df["Fanout"].values)
        return out

extractor = PhysicsExtractor(alpha=CFG.ALPHA)
df_feat   = extractor.extract(df)

PHY_FEATS = ["TM_N_ns","TF_N_ns","TM_P_ns","TF_P_ns", "d1R_N","d2R_N","d3R_N","dAP_N",
             "d1R_P","d2R_P","d3R_P","dAP_P", "CL_over_W","TIN_over_TM","Wn_Wp_ratio","log_FO"]

# FIXED RAW_FEATS: Added the 1-hot columns, and separated Vth_N_V / Vth_P_V
RAW_FEATS = ["Wn_um","Wp_um","L_um","VDD_V","TIN_ns","CLOAD_fF",
             "Temperature","Vth_N_V","Vth_P_V","N_series",
             "Gate_Type_INV", "Gate_Type_NAND2", "Gate_Type_NOR2",
             "Process_Corner_FF", "Process_Corner_FS", "Process_Corner_SF", 
             "Process_Corner_SS", "Process_Corner_TT"]

print(f"Physics features: {len(PHY_FEATS)} | Raw features: {len(RAW_FEATS)}")

Physics features: 16 | Raw features: 18


## Section 4 — PyTorch Data Pipeline

In [9]:
class TargetScaler:
    """
    Critical for Item 1: 
    Transforms nanoseconds -> log(picoseconds).
    This forces the model to focus on percentage error (MAPE) 
    rather than absolute error in the tiny ns values.
    """
    def fit(self, x): return self
    
    def transform(self, x): 
        # ns to ps (*1000), then log
        return np.log(np.clip(x * 1000.0, 1e-5, None))
    
    def inverse_transform(self, x): 
        # reverse log, then ps to ns (/1000)
        return np.exp(x) / 1000.0

class CMOSDataset(Dataset):
    def __init__(self, X_raw, X_phy, Y):
        self.X_raw = torch.tensor(X_raw, dtype=torch.float32)
        self.X_phy = torch.tensor(X_phy, dtype=torch.float32)
        self.Y     = torch.tensor(Y,     dtype=torch.float32)
    def __len__(self): return len(self.Y)
    def __getitem__(self, i): return self.X_raw[i], self.X_phy[i], self.Y[i]

def build_loaders(df: pd.DataFrame, cfg: Config):
    # Ensure no NaNs in features or targets
    df = df.dropna(subset=RAW_FEATS + PHY_FEATS + TARGETS).reset_index(drop=True)
    
    # 85% Train/Val, 15% Test
    trv, te  = train_test_split(df, test_size=cfg.TEST_FRAC, random_state=cfg.SEED)
    tr, val  = train_test_split(trv, test_size=cfg.VAL_FRAC/(1-cfg.TEST_FRAC), random_state=cfg.SEED)

    # Scalers
    sc_raw, sc_phy = StandardScaler(), StandardScaler()
    sc_y = TargetScaler()
    
    # Fit scalers only on training data
    sc_raw.fit(tr[RAW_FEATS].values.astype(np.float32))
    sc_phy.fit(tr[PHY_FEATS].values.astype(np.float32))
    
    def make_dataset(split):
        return CMOSDataset(
            sc_raw.transform(split[RAW_FEATS].values.astype(np.float32)),
            sc_phy.transform(split[PHY_FEATS].values.astype(np.float32)),
            sc_y.transform(split[TARGETS].values.astype(np.float32))
        )

    kw = dict(num_workers=0, pin_memory=cfg.USE_AMP)
    return (DataLoader(make_dataset(tr),  batch_size=cfg.BATCH, shuffle=True, **kw),
            DataLoader(make_dataset(val), batch_size=cfg.BATCH, shuffle=False, **kw),
            DataLoader(make_dataset(te),  batch_size=cfg.BATCH, shuffle=False, **kw),
            {"raw": sc_raw, "phy": sc_phy, "y": sc_y},
            {"train": tr, "val": val, "test": te},
            {"raw": len(RAW_FEATS), "phy": len(PHY_FEATS)})

dl_tr, dl_val, dl_te, scalers, splits, dims = build_loaders(df_feat, CFG)
RAW_DIM, PHY_DIM = dims["raw"], dims["phy"]
print("Data Loaders & TargetScaler ready ✓")

Data Loaders & TargetScaler ready ✓


## Section 5A — PhysicsResidualNet (PINN)

**Key Innovation — Residual Prediction:**
$$\hat{d} = \text{softplus}\!\left(d_{3\text{R}}^{\text{analytical}} + \Delta_{\text{NN}}\right)$$

The network learns *corrections* to the Three-Region analytical estimate.
At initialisation the NN correction ≈ 0, so predictions are immediately physically reasonable.

## Section 5A — PhysicsResidualNet (PINN) & Plain MLP

**Key Innovation — Residual Prediction:**
$$\hat{d} = d_{3\text{R}}^{\text{analytical}} + \Delta_{\text{NN}}$$

We also include a `PlainMLP` baseline (Action Item 4) to prove whether 
the residual architecture itself is causing issues compared to standard MLPs.

In [10]:
class ResBlock(nn.Module):
    """Pre-activation residual block: LayerNorm → GELU → Linear → drop → residual."""
    def __init__(self, in_d: int, out_d: int, drop: float = 0.15):
        super().__init__()
        self.norm1 = nn.LayerNorm(in_d)
        self.lin1  = nn.Linear(in_d, out_d)
        self.norm2 = nn.LayerNorm(out_d)
        self.lin2  = nn.Linear(out_d, out_d)
        self.drop  = nn.Dropout(drop)
        self.proj  = nn.Linear(in_d, out_d) if in_d != out_d else nn.Identity()

    def forward(self, x):
        h = self.drop(F.gelu(self.lin1(self.norm1(x))))
        return self.lin2(self.norm2(h)) + self.proj(x)

class PhysicsResidualNet(nn.Module):
    def __init__(self, raw_d, phy_d, hidden=None, drop=0.15, n_out=3):
        super().__init__()
        if hidden is None: hidden = [512, 256, 128, 64]
            
        self.raw_stem = nn.Sequential(nn.Linear(raw_d, hidden[0]),
                                      nn.LayerNorm(hidden[0]), nn.GELU())
        dims = [hidden[0]] + hidden
        self.raw_blocks = nn.ModuleList([ResBlock(dims[i], dims[i+1], drop) for i in range(len(dims)-1)])
            
        phy_h = hidden[1]
        self.phy_stem  = nn.Sequential(nn.Linear(phy_d, phy_h), nn.LayerNorm(phy_h), nn.GELU())
        self.phy_block = ResBlock(phy_h, phy_h // 2, drop)
        
        fuse_in = hidden[-1] + phy_h // 2
        self.fuse = nn.Sequential(nn.Linear(fuse_in, hidden[-1]), nn.LayerNorm(hidden[-1]), nn.GELU(), nn.Dropout(drop))
                                  
        self.heads = nn.ModuleList([nn.Sequential(nn.Linear(hidden[-1], 32), nn.GELU(), nn.Linear(32, 1)) for _ in range(n_out)])

    def forward(self, x_raw, x_phy):
        h = self.raw_stem(x_raw)
        for blk in self.raw_blocks: h = blk(h)
        p = self.phy_block(self.phy_stem(x_phy))
        h = self.fuse(torch.cat([h, p], dim=-1))
        return torch.cat([head(h) for head in self.heads], dim=-1)

class PlainMLP(nn.Module):
    """Standard MLP using BOTH Raw and Physics features."""
    def __init__(self, raw_d, phy_d, hidden=None, drop=0.15, n_out=3):
        super().__init__()
        hidden = hidden or [512, 256, 128, 64]
        in_d = raw_d + phy_d
        layers = []
        dims = [in_d] + hidden
        for i in range(len(dims)-1):
            layers += [nn.Linear(dims[i], dims[i+1]), nn.LayerNorm(dims[i+1]), nn.GELU(), nn.Dropout(drop)]
        self.net = nn.Sequential(*layers)
        self.heads = nn.ModuleList([nn.Linear(hidden[-1], 1) for _ in range(n_out)])

    def forward(self, x_raw, x_phy):
        x = torch.cat([x_raw, x_phy], dim=-1)
        h = self.net(x)
        return torch.cat([head(h) for head in self.heads], dim=-1)

class PlainMLP_NoPhysics(nn.Module):
    """Ablation Model: Standard MLP using ONLY Raw features (no physics features)."""
    def __init__(self, raw_d, hidden=None, drop=0.15, n_out=3):
        super().__init__()
        hidden = hidden or [512, 256, 128, 64]
        dims = [raw_d] + hidden
        layers = []
        for i in range(len(dims)-1):
            layers += [nn.Linear(dims[i], dims[i+1]), nn.LayerNorm(dims[i+1]), nn.GELU(), nn.Dropout(drop)]
        self.net = nn.Sequential(*layers)
        self.heads = nn.ModuleList([nn.Linear(hidden[-1], 1) for _ in range(n_out)])

    def forward(self, x_raw, x_phy):
        h = self.net(x_raw) # x_phy is strictly ignored
        return torch.cat([head(h) for head in self.heads], dim=-1)

# Instantiate models
pinn = PhysicsResidualNet(RAW_DIM, PHY_DIM, hidden=CFG.HIDDEN, drop=CFG.DROP)
plain_mlp = PlainMLP(RAW_DIM, PHY_DIM, hidden=CFG.HIDDEN, drop=CFG.DROP)
plain_mlp_no_phy = PlainMLP_NoPhysics(RAW_DIM, hidden=CFG.HIDDEN, drop=CFG.DROP)

print(f"PINN Params      : {sum(p.numel() for p in pinn.parameters()):,}")
print(f"Plain MLP Params : {sum(p.numel() for p in plain_mlp.parameters()):,}")
print(f"Ablation Params  : {sum(p.numel() for p in plain_mlp_no_phy.parameters()):,}")

PINN Params      : 1,078,979
Plain MLP Params : 192,515
Ablation Params  : 184,323


## Section 5B — FT-Transformer

Feature Tokenizer + Transformer (Gorishniy et al. 2021).
Each scalar feature → a learnable d-dimensional token; a `[CLS]` token aggregates.

In [11]:
class FeatureTokenizer(nn.Module):
    """x_i → token_i = x_i * W_i + b_i   (learnable per-feature affine map)"""
    def __init__(self, n_feat: int, d: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_feat, d))
        self.b = nn.Parameter(torch.zeros(n_feat, d))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))

    def forward(self, x):            # x: [B, F]
        return x.unsqueeze(-1) * self.W + self.b   # [B, F, d]


class FTTransformer(nn.Module):
    def __init__(self, raw_d, phy_d, d=64, n_heads=8, n_layers=4,
                 ffn_factor=2.0, drop=0.10, n_out=3):
        super().__init__()
        n_feat = raw_d + phy_d
        self.tok = FeatureTokenizer(n_feat, d)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        enc_lay  = nn.TransformerEncoderLayer(
            d_model=d, nhead=n_heads, dim_feedforward=int(d*ffn_factor),
            dropout=drop, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_lay, num_layers=n_layers)
        self.norm    = nn.LayerNorm(d)
        self.head    = nn.Sequential(
            nn.Linear(d, d//2), nn.GELU(), nn.Dropout(drop), nn.Linear(d//2, n_out))

    def forward(self, x_raw, x_phy):
        x   = torch.cat([x_raw, x_phy], dim=-1)
        tok = self.tok(x)
        cls = self.cls.expand(x.size(0), -1, -1)
        out = self.encoder(torch.cat([cls, tok], dim=1))
        
        # FIX: Softplus removed so the network can predict negative log values
        return self.head(self.norm(out[:, 0, :]))


ftnet = FTTransformer(RAW_DIM, PHY_DIM, d=CFG.FT_D, n_heads=CFG.FT_H,
                      n_layers=CFG.FT_L, drop=CFG.FT_DR)
print(f"FT-Transformer params  : {sum(p.numel() for p in ftnet.parameters()):,}")

FT-Transformer params  : 140,611


## Section 6 — Physics-Informed Loss

In [12]:
# ## Section 6 — Physics-Informed Loss
_RID = {k: i for i, k in enumerate(RAW_FEATS)}
IDX_CL  = _RID["CLOAD_fF"];  IDX_TIN = _RID["TIN_ns"]
IDX_VDD = _RID["VDD_V"];     IDX_WN  = _RID["Wn_um"]

class PhysicsInformedLoss(nn.Module):
    """
    Since targets are log-scaled, Smooth L1 Loss automatically minimizes Percentage Error.
    Physics Monotonicity Constraints ensure the derivatives match circuit laws.
    """
    def __init__(self, lam_phy=0.01, lam_sm=0.0):
        super().__init__()
        self.lam_phy = lam_phy
        self.lam_sm  = lam_sm

    def forward(self, pred, target, x_raw, x_phy, model):
        # 1. Huber Loss (Smooth L1) - extremely robust against outliers
        L = F.smooth_l1_loss(pred, target)

        # 2. Physics Monotonicity Constraints (Autograd)
        if self.lam_phy > 0 and model.training:
            sub  = min(128, x_raw.size(0))
            idx  = torch.randperm(x_raw.size(0), device=x_raw.device)[:sub]
            xr   = x_raw[idx].detach().requires_grad_(True)
            xp   = x_phy[idx].detach()
            g    = torch.autograd.grad(model(xr, xp).sum(), xr, create_graph=False)[0]
            
            # Physics rules: heavier load -> slower, higher VDD -> faster, etc.
            L_phy = (F.relu(-g[:, IDX_CL]).mean() + F.relu(-g[:, IDX_TIN]).mean() +
                     F.relu( g[:, IDX_VDD]).mean() + F.relu( g[:, IDX_WN]).mean())
            
            L += (self.lam_phy * 0.1) * L_phy

        if self.lam_sm > 0 and model.training:
            L += self.lam_sm * pred.var(dim=0).mean()

        return L

print("PhysicsInformedLoss defined  ✓")

PhysicsInformedLoss defined  ✓


## Section 7 — Training Pipeline  (AMP + DataParallel)

In [13]:
class EarlyStopping:
    def __init__(self, patience=20):
        self.p = patience; self.best = float("inf"); self.n = 0; self.stop = False
    def __call__(self, v):
        if v < self.best - 1e-5: self.best = v; self.n = 0
        else:
            self.n += 1
            if self.n >= self.p: self.stop = True
        return self.stop


def train_model(model, name: str, dl_tr, dl_val, cfg: Config):
    dev  = cfg.DEVICE
    model= model.to(dev)
    mdp  = nn.DataParallel(model) if cfg.USE_DP and cfg.N_GPUS>1 else model
    loss_fn = PhysicsInformedLoss(cfg.LAM_PHY, cfg.LAM_SM)
    opt  = AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WD)
    sch  = OneCycleLR(opt, max_lr=cfg.LR, steps_per_epoch=len(dl_tr),
                      epochs=cfg.EPOCHS, pct_start=0.1, anneal_strategy="cos")
    scaler = GradScaler(enabled=cfg.USE_AMP)
    es     = EarlyStopping(cfg.PATIENCE)
    tr_l, va_l, best_val, best_st = [], [], float("inf"), None

    n_params = sum(p.numel() for p in model.parameters())
    log.info(f"  Training {name} ({n_params:,} params) …")

    for ep in range(1, cfg.EPOCHS+1):
        mdp.train(); ep_loss = 0.0
        for xr, xp, y in dl_tr:
            xr, xp, y = xr.to(dev), xp.to(dev), y.to(dev)
            opt.zero_grad()
            with autocast(enabled=cfg.USE_AMP):
                base = mdp.module if hasattr(mdp,"module") else mdp
                pred = mdp(xr, xp)
                loss = loss_fn(pred, y, xr, xp, base)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP)
            scaler.step(opt); scaler.update(); sch.step()
            ep_loss += loss.item()

        mdp.eval(); v_loss = 0.0
        with torch.no_grad(), autocast(enabled=cfg.USE_AMP):
            for xr, xp, y in dl_val:
                pred   = mdp(xr.to(dev), xp.to(dev))
                v_loss += F.mse_loss(pred, y.to(dev)).item()

        tr_l.append(ep_loss/len(dl_tr)); va_l.append(v_loss/len(dl_val))
        if va_l[-1] < best_val:
            best_val = va_l[-1]
            best_st  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if ep % 10 == 0 or ep == 1:
            log.info(f"  Ep {ep:>4}/{cfg.EPOCHS}  "
                     f"train={tr_l[-1]:.5f}  val={va_l[-1]:.5f}  "
                     f"lr={sch.get_last_lr()[0]:.2e}")
        if es(va_l[-1]):
            log.info(f"  Early stop @ ep {ep}"); break

    model.load_state_dict(best_st)
    return model, tr_l, va_l


def predict(model, dl, cfg):
    model.eval(); dev = cfg.DEVICE; outs = []
    with torch.no_grad(), autocast(enabled=cfg.USE_AMP):
        for xr, xp, _ in dl:
            outs.append(model(xr.to(dev), xp.to(dev)).cpu().numpy())
    return np.vstack(outs)


print("train_model / predict defined  ✓")

train_model / predict defined  ✓


## Section 8 — Baseline Models  (XGBoost · LightGBM · RF)

In [14]:
def _Xsk(df):
    cols = [c for c in RAW_FEATS + PHY_FEATS if c in df.columns]
    return df[cols].fillna(0).values.astype(np.float32)

def train_baselines(splits, cfg):
    log.info("[8] Training sklearn baselines …")
    X_tr = _Xsk(splits["train"])
    
    # ACTION ITEM 1 FIX: Transform targets for Sklearn models using the TargetScaler
    # This trains XGBoost/LGBM on the exact same log-ps scale as the Neural Networks
    Y_tr = scalers["y"].transform(splits["train"][TARGETS].values)
    
    X_te = _Xsk(splits["test"])
    
    # Keep Test true targets raw (nanoseconds) for evaluation
    Y_te_raw = splits["test"][TARGETS].values 
    trained = {}

    log.info("  Random Forest …")
    rf = MultiOutputRegressor(
        RandomForestRegressor(n_estimators=500, n_jobs=-1,
                              random_state=cfg.SEED, max_depth=25), n_jobs=-1)
    rf.fit(X_tr, Y_tr)
    # Predict and reverse the scale back to nanoseconds
    trained["RandomForest"] = scalers["y"].inverse_transform(rf.predict(X_te))

    if HAS_XGB:
        log.info("  XGBoost …")
        preds = []
        for i in range(3):
            m = xgb.XGBRegressor(n_estimators=1500, max_depth=10,
                                  learning_rate=0.03, subsample=0.8,
                                  colsample_bytree=0.8, tree_method="hist",
                                  device="cuda" if torch.cuda.is_available() else "cpu",
                                  random_state=cfg.SEED, n_jobs=-1)
            m.fit(X_tr, Y_tr[:, i])
            preds.append(m.predict(X_te))
            
        # Predict and reverse the scale back to nanoseconds
        trained["XGBoost"] = scalers["y"].inverse_transform(np.column_stack(preds))

    if HAS_LGB:
        log.info("  LightGBM …")
        preds = []
        for i in range(3):
            m = lgb.LGBMRegressor(n_estimators=2000, max_depth=12, num_leaves=64,
                                   learning_rate=0.03, subsample=0.8,
                                   colsample_bytree=0.8, n_jobs=-1,
                                   random_state=cfg.SEED, verbose=-1)
            m.fit(X_tr, Y_tr[:, i])
            preds.append(m.predict(X_te))
            
        # Predict and reverse the scale back to nanoseconds
        trained["LightGBM"] = scalers["y"].inverse_transform(np.column_stack(preds))

    return trained, X_te, Y_te_raw

print("train_baselines defined  ✓")

train_baselines defined  ✓


## Section 9 — Evaluation & Comparison

In [15]:
def mape(yt, yp, eps=1e-10):
    return float(np.mean(np.abs((yt-yp)/(np.abs(yt)+eps)))*100)

def eval_model(name, y_true, y_pred):
    res = {"Model": name}
    for i, t in enumerate(TARGETS):
        yt, yp = y_true[:, i], y_pred[:, i]
        res[f"MAE_{t}"]  = mean_absolute_error(yt, yp)
        res[f"RMSE_{t}"] = mean_squared_error(yt, yp)**0.5
        res[f"MAPE_{t}"] = mape(yt, yp)
        res[f"R2_{t}"]   = r2_score(yt, yp)
    res["MAE_avg"]  = np.mean([res[f"MAE_{t}"]  for t in TARGETS])
    res["RMSE_avg"] = np.mean([res[f"RMSE_{t}"] for t in TARGETS])
    res["MAPE_avg"] = np.mean([res[f"MAPE_{t}"] for t in TARGETS])
    res["R2_avg"]   = np.mean([res[f"R2_{t}"]   for t in TARGETS])
    return res

def eval_analytical(test_df):
    y_true = test_df[TARGETS].values
    results = []
    for label, fc, rc in [
        ("1-Region",   "d1R_N","d1R_P"),
        ("2-Region",   "d2R_N","d2R_P"),
        ("3-Region",   "d3R_N","d3R_P"),
        ("Alpha-Power","dAP_N","dAP_P"),
    ]:
        avg  = (test_df[rc].values + test_df[fc].values) / 2
        pred = np.column_stack([test_df[rc].values, test_df[fc].values, avg])
        results.append(eval_model(label, y_true, pred))
    return results

print("Evaluation helpers defined  ✓")

Evaluation helpers defined  ✓


## Section 10 — Visualisation (10 Plots)

In [16]:
def make_plots(df_feat, splits, all_preds, all_results, hist, cfg):
    D = cfg.OUT_DIR + "/plots"
    os.makedirs(D, exist_ok=True)

    # ── 1. Training curves ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, len(hist), figsize=(6*len(hist), 4))
    if len(hist) == 1: axes = [axes]
    for ax, (nm, (tr, va)) in zip(axes, hist.items()):
        ax.plot(tr, label="Train", color=PALETTE.get(nm,"#666"))
        ax.plot(va, label="Val",   color=PALETTE.get(nm,"#666"), ls="--", alpha=0.7)
        ax.set_title(f"{nm} Loss"); ax.set_yscale("log"); ax.legend()
    fig.tight_layout(); fig.savefig(f"{D}/01_training_curves.png",dpi=110,bbox_inches="tight")
    plt.show()

    # ── 2–3. Pred vs Actual (PINN + FT) ──────────────────────────────────
    for nm, yt, yp in [(k, splits["test"][TARGETS].values, v)
                        for k, v in all_preds.items() if k in ("PINN","FT-Transformer")]:
        fig, axes = plt.subplots(1,3,figsize=(14,4))
        for ax, i, t in zip(axes, range(3), ["Rise","Fall","Avg"]):
            vmin,vmax = min(yt[:,i].min(),yp[:,i].min()),max(yt[:,i].max(),yp[:,i].max())
            ax.scatter(yt[:,i], yp[:,i], alpha=0.25, s=5, color=PALETTE.get(nm,"#333"))
            ax.plot([vmin,vmax],[vmin,vmax],"r--",lw=1.5)
            ax.set_xlabel(f"True {t} (ns)"); ax.set_ylabel(f"Pred {t} (ns)")
            ax.set_title(f"{nm}  {t}\nR²={r2_score(yt[:,i],yp[:,i]):.4f} "
                         f"MAPE={mape(yt[:,i],yp[:,i]):.1f}%")
        fig.tight_layout()
        fig.savefig(f"{D}/02_pred_actual_{nm.replace(' ','-')}.png",dpi=110,bbox_inches="tight")
        plt.show()

    # ── 4. Delay vs Input Slew ────────────────────────────────────────────
    sub = df_feat.query("CLOAD_fF < 50 and Process_Corner=='TT'").sort_values("TIN_ns")
    fig, ax = plt.subplots(figsize=(7,4))
    ax.scatter(sub.TIN_ns*1000, sub.avg_delay_ns*1000, alpha=0.2, s=5, label="Simulated")
    ax.plot(sub.TIN_ns*1000, sub.d3R_N*1000, "r-", lw=2, label="3-Region (paper)")
    ax.set_xlabel("TIN (ps)"); ax.set_ylabel("Avg Delay (ps)")
    ax.set_title("Delay vs Input Slew (TT, CLOAD<50fF)"); ax.legend()
    fig.tight_layout(); fig.savefig(f"{D}/04_delay_vs_TIN.png",dpi=110,bbox_inches="tight")
    plt.show()

    # ── 5. Delay vs CLOAD ────────────────────────────────────────────────
    g  = df_feat.groupby(pd.cut(df_feat.CLOAD_fF,20)).avg_delay_ns.mean()
    xs = [iv.mid for iv in g.index]
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(xs, [v*1000 for v in g.values], "-o", color="#2A9D8F")
    ax.set_xlabel("CLOAD (fF)"); ax.set_ylabel("Mean Avg Delay (ps)")
    ax.set_title("Delay vs Load Capacitance")
    fig.tight_layout(); fig.savefig(f"{D}/05_delay_vs_CLOAD.png",dpi=110,bbox_inches="tight")
    plt.show()

    # ── 6. Delay vs VDD ──────────────────────────────────────────────────
    g  = df_feat.groupby(pd.cut(df_feat.VDD_V,10)).avg_delay_ns.mean()
    xs = [iv.mid for iv in g.index]
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(xs, [v*1000 for v in g.values], "-o", color="#E63946")
    ax.set_xlabel("VDD (V)"); ax.set_ylabel("Mean Avg Delay (ps)")
    ax.set_title("Delay vs Supply Voltage")
    fig.tight_layout(); fig.savefig(f"{D}/06_delay_vs_VDD.png",dpi=110,bbox_inches="tight")
    plt.show()

    # ── 7. Delay vs Temperature ──────────────────────────────────────────
    g  = df_feat.groupby(pd.cut(df_feat.Temperature,15)).avg_delay_ns.mean()
    xs = [iv.mid for iv in g.index]
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(xs, [v*1000 for v in g.values], "-o", color="#457B9D")
    ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Mean Avg Delay (ps)")
    ax.set_title("Delay vs Temperature")
    fig.tight_layout(); fig.savefig(f"{D}/07_delay_vs_temp.png",dpi=110,bbox_inches="tight")
    plt.show()

    # ── 8. Gate type comparison ──────────────────────────────────────────
    g  = df_feat.groupby("Gate_Type").avg_delay_ns.mean()*1000
    fig, ax = plt.subplots(figsize=(6,4))
    g.plot(kind="bar", ax=ax, color=["#E63946","#2A9D8F","#457B9D"], edgecolor="white")
    ax.set_xlabel("Gate Type"); ax.set_ylabel("Mean Avg Delay (ps)")
    ax.set_title("Mean Delay by Gate Type")
    fig.tight_layout(); fig.savefig(f"{D}/08_gate_type.png",dpi=110,bbox_inches="tight")
    plt.show()

    # ── 9. Model comparison ──────────────────────────────────────────────
    df_r = pd.DataFrame(all_results).sort_values("MAPE_avg")
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, m in zip(axes, ["MAE_avg","MAPE_avg","R2_avg"]):
        colors = [PALETTE.get(n,"#888") for n in df_r["Model"]]
        ax.barh(df_r["Model"], df_r[m], color=colors, edgecolor="white")
        ax.set_title(m); ax.invert_yaxis()
    fig.suptitle("Model Comparison", fontsize=13)
    fig.tight_layout(); fig.savefig(f"{D}/09_model_comparison.png",dpi=110,bbox_inches="tight")
    plt.show()

    # ── 10. Error distribution ───────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9,4))
    y_true = splits["test"][TARGETS].values
    for nm, yp in all_preds.items():
        if nm in ("PINN","FT-Transformer","XGBoost","3-Region"):
            err = np.abs(y_true[:,2]-yp[:,2])/(np.abs(y_true[:,2])+1e-10)*100
            ax.hist(err, bins=60, alpha=0.5, label=nm, density=True,
                    color=PALETTE.get(nm,"#888"))
    ax.set_xlabel("Absolute % Error"); ax.set_title("Error Distribution — Avg Delay")
    ax.set_xlim(0,80); ax.legend()
    fig.tight_layout(); fig.savefig(f"{D}/10_error_dist.png",dpi=110,bbox_inches="tight")
    plt.show()

    print(f"All plots saved to {D}/")

print("make_plots defined  ✓")

make_plots defined  ✓


## Section 11 — SHAP Explainability

In [17]:
def run_shap(xgb_models, splits, cfg):
    if not HAS_SHAP:
        print("SHAP not installed — pip install shap"); return
    if not HAS_XGB or xgb_models is None:
        print("Need XGBoost for SHAP"); return

    te    = splits["test"].head(1000)
    fcols = [c for c in RAW_FEATS + PHY_FEATS if c in te.columns]
    X_te  = te[fcols].fillna(0).values.astype(np.float32)

    # avg_delay model (index 2)
    predictor = xgb_models[2] if isinstance(xgb_models, list) else xgb_models

    exp  = shap.TreeExplainer(predictor)
    sv   = exp.shap_values(X_te[:500])
    D    = cfg.OUT_DIR + "/plots"

    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv, X_te[:500], feature_names=fcols, show=False, max_display=20)
    plt.title("SHAP Summary — XGBoost (avg_delay)")
    plt.tight_layout()
    plt.savefig(f"{D}/11_shap_summary.png", dpi=110, bbox_inches="tight"); plt.show()

    mean_sv = np.abs(sv).mean(0)
    top     = np.argsort(mean_sv)[-20:][::-1]
    plt.figure(figsize=(9, 6))
    plt.barh([fcols[i] for i in top], mean_sv[top], color="#2A9D8F")
    plt.xlabel("Mean |SHAP|"); plt.title("Feature Importance (SHAP)")
    plt.tight_layout()
    plt.savefig(f"{D}/12_shap_bar.png", dpi=110, bbox_inches="tight"); plt.show()
    print("SHAP plots saved.")

print("run_shap defined  ✓")

run_shap defined  ✓


In [18]:
def run_cross_corner_test(df_full, cfg):
    """
    Train on TT/FF/SS, Test on unseen FS/SF.
    FIX: Exclude Process_Corner one-hot features.
    Reason: FS/SF are always 0 in training. StandardScaler assigns
    scale=1, mean=0. When test rows have value=1, the model receives
    an out-of-distribution input it has never seen → MAPE=96%.
    Solution: rely on Vth_N_V and Vth_P_V (continuous, in-range) and
    physics features to encode the corner physics.
    """
    print("\n" + "═"*65)
    print("  CROSS-CORNER GENERALIZATION TEST (Train: TT/FF/SS | Test: FS/SF)")
    print("═"*65)

    train_mask = df_full['Process_Corner'].isin(['TT', 'FF', 'SS'])
    test_mask  = df_full['Process_Corner'].isin(['FS', 'SF'])
    df_trv = df_full[train_mask].reset_index(drop=True)
    df_te  = df_full[test_mask].reset_index(drop=True)
    df_tr, df_val = train_test_split(df_trv, test_size=0.1, random_state=cfg.SEED)

    # ── KEY FIX: remove one-hot corner features ───────────────────────────
    # These are always 0 for FS/SF in training, causing out-of-distribution
    # inputs during test. Vth_N_V and Vth_P_V encode the same information
    # continuously and generalise correctly.
    cross_raw_feats = [f for f in RAW_FEATS
                       if not f.startswith('Process_Corner_')]
    cross_raw_dim   = len(cross_raw_feats)
    print(f"  Features used: {len(RAW_FEATS)} → {cross_raw_dim} (removed corner one-hots)")
    print(f"  Corner info encoded by: Vth_N_V, Vth_P_V, and all physics features")
    # ─────────────────────────────────────────────────────────────────────

    sc_raw, sc_phy, sc_y = StandardScaler(), StandardScaler(), TargetScaler()
    sc_raw.fit(df_tr[cross_raw_feats].values.astype(np.float32))
    sc_phy.fit(df_tr[PHY_FEATS].values.astype(np.float32))

    def make_ds(split):
        return CMOSDataset(
            sc_raw.transform(split[cross_raw_feats].values.astype(np.float32)),
            sc_phy.transform(split[PHY_FEATS].values.astype(np.float32)),
            sc_y.transform(split[TARGETS].values.astype(np.float32))
        )

    dlt = DataLoader(make_ds(df_tr),  batch_size=cfg.BATCH, shuffle=True)
    dlv = DataLoader(make_ds(df_val), batch_size=cfg.BATCH, shuffle=False)
    dle = DataLoader(make_ds(df_te),  batch_size=cfg.BATCH, shuffle=False)

    # Build model with the smaller input dimension
    model = FTTransformer(cross_raw_dim, PHY_DIM,
                          d=cfg.FT_D, n_heads=cfg.FT_H,
                          n_layers=cfg.FT_L, drop=cfg.FT_DR)
    trained_model, _, _ = train_model(model, "Cross-Corner FT-T", dlt, dlv, cfg)

    preds_ns  = sc_y.inverse_transform(predict(trained_model, dle, cfg))
    y_true_ns = df_te[TARGETS].values

    res = eval_model("FT-T (Unseen FS/SF)", y_true_ns, preds_ns)
    print("\n" + "═"*65)
    print("  RESULTS")
    print("═"*65)
    import pandas as pd
    display(pd.DataFrame([res])[["Model", "MAE_avg", "MAPE_avg", "R2_avg"]])
    return res

In [19]:
# --- Update your report_granular_metrics in Section 11/12 ---

def report_granular_metrics(model_name, y_true_ns, y_pred_ns, test_df):
    results = []
    test_df = test_df.reset_index(drop=True)
    
    groups = {
        "Gate: INV": test_df['Gate_Type_INV'] == 1.0,
        "Gate: NAND2": test_df['Gate_Type_NAND2'] == 1.0,
        "Gate: NOR2": test_df['Gate_Type_NOR2'] == 1.0,
        "Corner: SS": test_df['Process_Corner_SS'] == 1.0,
        "Corner: FF": test_df['Process_Corner_FF'] == 1.0
    }
    
    print(f"\n--- Granular Analysis for Paper Table: {model_name} ---")
    for label, mask in groups.items():
        if mask.any():
            m = eval_model(label, y_true_ns[mask], y_pred_ns[mask])
            results.append({"Category": label, "MAPE": f"{m['MAPE_avg']:.2f}%", "R2": f"{m['R2_avg']:.4f}"})
    
    display(pd.DataFrame(results))

## Section 12 — Run Everything  🚀

In [20]:
def main():
    t0 = time.time()
    log.info("═"*65)
    log.info("  CMOS Delay Predictor — Physics-Informed ML Framework (FINAL)")
    log.info("═"*65)

    all_results = []; all_preds = {}; hist = {}
    y_true_test_ns = splits["test"][TARGETS].values

    # 1. PhysicsResidualNet (PINN)
    log.info("\n[1] Training PINN...")
    pinn_m, tr_l, va_l = train_model(pinn, "PINN", dl_tr, dl_val, CFG)
    hist["PINN"] = (tr_l, va_l)
    pinn_pred_ns = scalers["y"].inverse_transform(predict(pinn_m, dl_te, CFG)) 
    all_preds["PINN"] = pinn_pred_ns
    all_results.append(eval_model("PINN", y_true_test_ns, pinn_pred_ns))
    report_granular_metrics("PINN", y_true_test_ns, pinn_pred_ns, splits["test"])
    
    # 2. Plain MLP (Physics-Informed)
    log.info("\n[2] Training Plain MLP (With Physics Features)...")
    mlp_m, tr_lm, va_lm = train_model(plain_mlp, "Plain MLP", dl_tr, dl_val, CFG)
    hist["Plain MLP"] = (tr_lm, va_lm)
    mlp_pred_ns = scalers["y"].inverse_transform(predict(mlp_m, dl_te, CFG)) 
    all_preds["Plain MLP"] = mlp_pred_ns
    all_results.append(eval_model("Plain MLP", y_true_test_ns, mlp_pred_ns))

    # 3. [TASK 2] Ablation Model (No Physics)
    log.info("\n[3] Training Ablation Study (Raw Features Only)...")
    abl_m, tr_la, va_la = train_model(plain_mlp_no_phy, "Ablation (No Physics)", dl_tr, dl_val, CFG)
    hist["Ablation"] = (tr_la, va_la)
    abl_pred_ns = scalers["y"].inverse_transform(predict(abl_m, dl_te, CFG)) 
    all_results.append(eval_model("Ablation (No Physics)", y_true_test_ns, abl_pred_ns))

    # 4. FT-Transformer
    log.info("\n[4] Training FT-Transformer...")
    ft_m, tr_l2, va_l2 = train_model(ftnet, "FT-Transformer", dl_tr, dl_val, CFG)
    hist["FT-Transformer"] = (tr_l2, va_l2)
    ft_pred_ns = scalers["y"].inverse_transform(predict(ft_m, dl_te, CFG))
    all_preds["FT-Transformer"] = ft_pred_ns
    all_results.append(eval_model("FT-Transformer", y_true_test_ns, ft_pred_ns))
    report_granular_metrics("FT-Transformer", y_true_test_ns, ft_pred_ns, splits["test"])

    # --- ADDED: Cross-corner performance evaluation ---
    # From the main 5-corner model, evaluate specifically on FS/SF rows
    fs_sf_mask = splits["test"]["Process_Corner"].isin(["FS", "SF"])
    if fs_sf_mask.sum() > 0:
        res_cc = eval_model(
            "FT-T on FS/SF corners",
            y_true_test_ns[fs_sf_mask],
            ft_pred_ns[fs_sf_mask]
        )
        print("\n--- Cross-Corner (FS/SF) from main model ---")
        print(f"MAPE: {res_cc['MAPE_avg']:.2f}%  R²: {res_cc['R2_avg']:.4f}")
    # --------------------------------------------------

    # 5. Baselines (XGB/LGB/RF)
    baselines, _, _ = train_baselines(splits, CFG)
    for nm, pred_ns in baselines.items():
        all_preds[nm] = pred_ns
        all_results.append(eval_model(nm, y_true_test_ns, pred_ns))

    # 6. Analytical Model Benchmarks
    log.info("\n[6] Evaluating Analytical Baselines...")
    test_df_anal = df_feat.iloc[splits["test"].index].reset_index(drop=True)
    analytic_models = {"1-Region": ("d1R_P", "d1R_N"), "2-Region": ("d2R_P", "d2R_N"), 
                       "3-Region": ("d3R_P", "d3R_N"), "Alpha-Power": ("dAP_P", "dAP_N")}
    for mod, (rc, fc) in analytic_models.items():
        pred_ns = np.column_stack([test_df_anal[rc], test_df_anal[fc], (test_df_anal[rc]+test_df_anal[fc])/2])
        all_results.append(eval_model(mod, y_true_test_ns, pred_ns))

    # --- RESULTS SUMMARY ---
    df_res = pd.DataFrame(all_results).sort_values("MAPE_avg")
    print("\n" + "═"*65)
    print("  FINAL MODEL COMPARISON (Ablation Included)")
    print("═"*65)
    display(df_res[["Model","MAE_avg","MAPE_avg","R2_avg"]].round(4))
    
    # 7. [REMOVED/REPLACED] Cross-Corner Generalization Test
    # This is now handled by the evaluation above per instructions
    # run_cross_corner_test(df_feat, CFG)

    # 8. SHAP & Plots
    if HAS_XGB:
        X_tr = _Xsk(splits["train"])
        Y_tr_log = scalers["y"].transform(splits["train"][TARGETS].values)
        m_shap = xgb.XGBRegressor(n_estimators=500, max_depth=6, random_state=42)
        m_shap.fit(X_tr, Y_tr_log[:, 2])
        run_shap(m_shap, splits, CFG)

    make_plots(df_feat, splits, all_preds, all_results, hist, CFG)
    
    elapsed = (time.time() - t0)/60
    log.info(f"Process Complete in {elapsed:.1f} mins.")

main()

2026-06-14 06:20:59,214 [INFO] ═════════════════════════════════════════════════════════════════
2026-06-14 06:20:59,214 [INFO]   CMOS Delay Predictor — Physics-Informed ML Framework (FINAL)
2026-06-14 06:20:59,215 [INFO] ═════════════════════════════════════════════════════════════════
2026-06-14 06:20:59,217 [INFO] 
[1] Training PINN...
2026-06-14 06:21:02,968 [INFO]   Training PINN (1,078,979 params) …
2026-06-14 06:21:05,591 [INFO]   Ep    1/150  train=2.40583  val=8.73260  lr=5.05e-05
2026-06-14 06:21:12,715 [INFO]   Ep   10/150  train=0.08421  val=0.12812  lr=7.62e-04
2026-06-14 06:21:20,742 [INFO]   Ep   20/150  train=0.02349  val=0.11643  lr=9.97e-04
2026-06-14 06:21:28,877 [INFO]   Ep   30/150  train=0.01807  val=0.08286  lr=9.70e-04
2026-06-14 06:21:36,873 [INFO]   Ep   40/150  train=0.01159  val=0.09189  lr=9.18e-04
2026-06-14 06:21:42,497 [INFO]   Early stop @ ep 47



--- Granular Analysis for Paper Table: PINN ---


,Category,MAPE,R2
0,Gate: INV,20.38%,0.9066
1,Gate: NAND2,15.06%,0.9186
2,Gate: NOR2,22.73%,0.3262
3,Corner: SS,18.88%,0.4793
4,Corner: FF,17.72%,0.4680


2026-06-14 06:21:42,558 [INFO] 
[2] Training Plain MLP (With Physics Features)...
2026-06-14 06:21:42,561 [INFO]   Training Plain MLP (192,515 params) …
2026-06-14 06:21:43,017 [INFO]   Ep    1/150  train=2.15999  val=5.58337  lr=5.05e-05
2026-06-14 06:21:46,782 [INFO]   Ep   10/150  train=0.13330  val=0.17346  lr=7.62e-04
2026-06-14 06:21:51,011 [INFO]   Ep   20/150  train=0.06495  val=0.08052  lr=9.97e-04
2026-06-14 06:21:55,292 [INFO]   Ep   30/150  train=0.04839  val=0.04626  lr=9.70e-04
2026-06-14 06:21:59,257 [INFO]   Ep   40/150  train=0.04231  val=0.04371  lr=9.18e-04
2026-06-14 06:22:03,360 [INFO]   Ep   50/150  train=0.03736  val=0.03529  lr=8.43e-04
2026-06-14 06:22:07,561 [INFO]   Ep   60/150  train=0.03309  val=0.02776  lr=7.50e-04
2026-06-14 06:22:11,853 [INFO]   Ep   70/150  train=0.03039  val=0.02119  lr=6.43e-04
2026-06-14 06:22:16,361 [INFO]   Ep   80/150  train=0.02781  val=0.01955  lr=5.29e-04
2026-06-14 06:22:20,618 [INFO]   Ep   90/150  train=0.02582  val=0.02205 


--- Granular Analysis for Paper Table: FT-Transformer ---


...


═════════════════════════════════════════════════════════════════
  FINAL MODEL COMPARISON (Ablation Included)
═════════════════════════════════════════════════════════════════


,Model,MAE_avg,MAPE_avg,R2_avg
1,Plain MLP,0.0017,5.0286,0.9710
2,Ablation (No Physics),0.0018,5.2970,0.9634
3,FT-Transformer,0.0025,6.4950,0.9685
5,XGBoost,0.0025,6.5040,0.8234
6,LightGBM,0.0029,6.6951,0.8153
4,RandomForest,0.0037,8.6751,0.6860
0,PINN,0.0095,18.7376,0.4867
7,1-Region,0.0333,165.3626,-0.5555
8,2-Region,0.0925,538.3782,-4.5051
9,3-Region,0.1223,667.4061,-8.7828


SHAP plots saved.


2026-06-14 06:27:37,569 [INFO] Process Complete in 6.6 mins.


All plots saved to cmos_output/plots/


## Section 13 — Flask REST API

Save the code below as `flask_api.py` and run:
```bash
pip install flask flask-cors
python flask_api.py
```

In [21]:
flask_api_code = '''
#!/usr/bin/env python3
"""
CMOS Delay Predictor — Flask REST API
POST /predict  →  {"Wn_um":1.6,"Wp_um":3.2,"L_um":0.045,"VDD_V":1.2,
                   "TIN_ns":0.01,"CLOAD_fF":10,"CW_fF":0,"RW_ohm":0,
                   "Fanout":4,"N_series":1,"Temperature":27,
                   "Gate_Type":"INV","Process_Corner":"TT"}
"""
import os, pickle, math, json, sys
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from flask import Flask, request, jsonify
from flask_cors import CORS

# ─── Load your trained model (run main() first) ───────────────────────────────
sys.path.insert(0, ".")
# Import classes from the notebook (copy them here or import from a .py file)
# ...
# MODEL = PhysicsResidualNet(RAW_DIM, PHY_DIM, ...)
# MODEL.load_state_dict(torch.load("cmos_output/models/pinn.pt"))
# MODEL.eval()
# with open("cmos_output/models/scalers.pkl","rb") as f: SCALERS = pickle.load(f)

app = Flask(__name__); CORS(app)

@app.route("/health")
def health(): return jsonify({"status":"ok"})

@app.route("/predict", methods=["POST"])
def predict_route():
    data = request.get_json()
    # 1. adapt_dataset(pd.DataFrame([data]))
    # 2. PhysicsExtractor().extract(df)
    # 3. scaler.transform + MODEL inference
    # 4. return predicted delays
    return jsonify({"error": "copy model classes here first"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

# Save to file
with open(f"{CFG.OUT_DIR}/flask_api.py", "w") as f:
    f.write(flask_api_code)
print(f"Flask API saved to {CFG.OUT_DIR}/flask_api.py")

Flask API saved to cmos_output/flask_api.py


## Section 14 — Streamlit Dashboard

Save as `streamlit_app.py` and run:
```bash
pip install streamlit plotly
streamlit run streamlit_app.py
```
The full `streamlit_app.py` is already in your downloaded files.  
It provides interactive sliders, real-time PINN prediction, model comparison,
sensitivity sweeps, and corner analysis — all in a browser-based dashboard.

In [22]:
# Quick local test — launch Streamlit (comment out on Kaggle)
# import subprocess
# subprocess.Popen(["streamlit", "run", "streamlit_app.py", "--server.port=8501"])
# print("Streamlit running → http://localhost:8501")
print("To run the dashboard locally:")
print("  pip install streamlit plotly flask flask-cors")
print("  streamlit run streamlit_app.py")

To run the dashboard locally:
  pip install streamlit plotly flask flask-cors
  streamlit run streamlit_app.py


**Save your Model and Scalers to the "Output" Folder**

In [23]:
import os
import pickle
import torch

# Create the directory
os.makedirs('/kaggle/working/sta_engine_data', exist_ok=True)

# 1. Save the FT-Transformer (using the global variable name 'ftnet')
# Make sure you have already run the main() function!
if 'ftnet' in locals() or 'ftnet' in globals():
    torch.save(ftnet.state_dict(), '/kaggle/working/sta_engine_data/trained_model.pth')
    print("Model 'ftnet' saved successfully.")
else:
    print("Error: 'ftnet' not found. Did you run the training cell?")

# 2. Save the Scalers (using the global variable 'scalers')
if 'scalers' in locals() or 'scalers' in globals():
    with open('/kaggle/working/sta_engine_data/scalers.pkl', 'wb') as f:
        pickle.dump(scalers, f)
    print("Scalers saved successfully.")
else:
    print("Error: 'scalers' not found.")

Model 'ftnet' saved successfully.
Scalers saved successfully.


**The Gate Class**

In [24]:
# class Gate:
#     def __init__(self, name, gate_type, wn_um, wp_um, vdd=1.0, temp=25.0, corner="TT"):
#         self.name = name
#         self.gate_type = gate_type # 'INV', 'NAND2', or 'NOR2'
#         self.wn_um = wn_um
#         self.wp_um = wp_um
#         self.vdd = vdd
#         self.temp = temp
#         self.corner = corner
        
#         # Vth mapping based on corner
#         vth_map = {
#             "TT": (0.32, 0.30),
#             "FF": (0.29, 0.27),
#             "SS": (0.35, 0.33),
#             "FS": (0.29, 0.33),
#             "SF": (0.35, 0.27)
#         }
#         self.vth_n, self.vth_p = vth_map.get(corner, (0.32, 0.30))
        
#         # Connections
#         self.fanin = []  # List of Gate objects that drive this gate
#         self.fanout = [] # List of Gate objects this gate drives
        
#         # Timing state (to be filled by the ML model later)
#         self.arrival_time = 0.0
#         self.output_slew = 10.0 # Initial slew in ps
#         self.delay = 0.0
#         self.cload_fF = 0.0

#     def get_cin(self):
#         """
#         Physics Model: Gate capacitance + Realistic Wire Routing Parasitics.
#         """
#         ff_per_um = 1.0 
#         wire_cap_fF = 1.5 # Assume 1.5 fF of metal wire routing between gates
        
#         if self.gate_type == "INV":
#             return wire_cap_fF + (self.wn_um + self.wp_um) * ff_per_um
#         elif self.gate_type == "NAND2":
#             # NAND2 has 2 NMOS in series and 2 PMOS in parallel
#             return wire_cap_fF + (2 * self.wn_um + self.wp_um) * ff_per_um
#         elif self.gate_type == "NOR2":
#             return wire_cap_fF + (self.wn_um + 2 * self.wp_um) * ff_per_um
#         return wire_cap_fF + 0.5 
    
#     def __repr__(self):
#         return f"<Gate {self.name} ({self.gate_type})>"

In [25]:
class Gate:
    def __init__(self, name, gate_type, wn_um, wp_um, vdd=1.0, temp=25.0, corner="TT"):
        self.name = name; self.gate_type = gate_type
        self.wn_um = wn_um; self.wp_um = wp_um
        self.vdd = vdd; self.temp = temp; self.corner = corner
        
        vth_map = {"TT": (0.32, 0.30), "FF": (0.29, 0.27), "SS": (0.35, 0.33), "FS": (0.29, 0.33), "SF": (0.35, 0.27)}
        self.vth_n, self.vth_p = vth_map.get(corner, (0.32, 0.30))
        self.fanin = []; self.fanout = []
        self.arrival_time = 0.0; self.output_slew = 10.0
        self.delay = 0.0; self.cload_fF = 0.0

    def get_cin(self):
        # Increased to 3.0 fF to match real 45nm standard cell routing interconnects
        wire_cap_fF = 3.0 
        ff_per_um = 1.0 
        if self.gate_type == "INV": return wire_cap_fF + (self.wn_um + self.wp_um) * ff_per_um
        elif self.gate_type == "NAND2": return wire_cap_fF + (2 * self.wn_um + self.wp_um) * ff_per_um
        elif self.gate_type == "NOR2": return wire_cap_fF + (self.wn_um + 2 * self.wp_um) * ff_per_um
        return wire_cap_fF + 0.5 

class Circuit:
    def __init__(self):
        self.gates = {}; self.primary_inputs = []

    def add_gate(self, gate):
        self.gates[gate.name] = gate

    def connect(self, driver_name, load_name):
        driver = self.gates[driver_name]; load = self.gates[load_name]
        driver.fanout.append(load); load.fanin.append(driver)

    def calculate_all_cloads(self):
        for name, gate in self.gates.items():
            if not gate.fanout:
                gate.cload_fF = 15.0 # REALISTIC output pad capacitance for SPICE
            else:
                gate.cload_fF = sum(child.get_cin() for child in gate.fanout)

**Create the Circuit Manager**

In [26]:
# class Circuit:
#     def __init__(self):
#         self.gates = {} # name -> Gate object
#         self.primary_inputs = []

#     def add_gate(self, gate):
#         self.gates[gate.name] = gate

#     def connect(self, driver_name, load_name):
#         """Creates a wire between two gates."""
#         driver = self.gates[driver_name]
#         load = self.gates[load_name]
#         driver.fanout.append(load)
#         load.fanin.append(driver)

#     def calculate_all_cloads(self):
#         """
#         The Cload of a gate is the sum of the Cins of all gates it drives.
#         """
#         for name, gate in self.gates.items():
#             if not gate.fanout:
#                 gate.cload_fF = 5.0 # Assume 5fF load for primary outputs
#             else:
#                 gate.cload_fF = sum(child.get_cin() for child in gate.fanout)

**Test with the C17 Benchmark**

In [27]:
# 1. Create the circuit
c17 = Circuit()

# 2. Add Gates (Physical parameters are placeholders for now)
# Gates: G10, G11, G16, G19, G22, G23 are all NAND2 in C17
gate_names = ["G10", "G11", "G16", "G19", "G22", "G23"]
for name in gate_names:
    c17.add_gate(Gate(name, "NAND2", wn_um=0.12, wp_um=0.24))

# 3. Add Primary Inputs (Treat them as simple gates for now)
inputs = ["N1", "N2", "N3", "N6", "N7"]
for name in inputs:
    c17.add_gate(Gate(name, "INV", wn_um=0.12, wp_um=0.24))

# 4. Define the Connections (The 'Wires')
wires = [
    ("N1", "G10"), ("N3", "G10"),
    ("N3", "G11"), ("N6", "G11"),
    ("G10", "G22"), ("G11", "G16"),
    ("N2", "G16"), ("G11", "G19"),
    ("N7", "G19"), ("G16", "G22"),
    ("G19", "G23"), ("G22", "G23")
]

for driver, load in wires:
    c17.connect(driver, load)

# 5. Calculate Loads
c17.calculate_all_cloads()

# 6. Verify
print(f"Total gates in circuit: {len(c17.gates)}")
print(f"Load capacitance for G11: {c17.gates['G11'].cload_fF:.3f} fF")
print(f"G11 drives: {[g.name for g in c17.gates['G11'].fanout]}")

Total gates in circuit: 11
Load capacitance for G11: 6.960 fF
G11 drives: ['G16', 'G19']


**Topological Sort**

In [28]:
from collections import deque

def get_topological_order(circuit):
    # Count how many inputs each gate has (in-degree)
    in_degree = {name: len(gate.fanin) for name, gate in circuit.gates.items()}
    
    # Start with gates that have NO inputs (Primary Inputs)
    queue = deque([name for name, deg in in_degree.items() if deg == 0])
    order = []
    
    while queue:
        u_name = queue.popleft()
        u_gate = circuit.gates[u_name]
        order.append(u_gate)
        
        # Look at the gates this gate drives
        for v_gate in u_gate.fanout:
            in_degree[v_gate.name] -= 1
            # If all inputs of the load are processed, add to queue
            if in_degree[v_gate.name] == 0:
                queue.append(v_gate.name)
                
    return order

# Test the order
sorted_gates = get_topological_order(c17)
print("Processing Order:", [g.name for g in sorted_gates])

Processing Order: ['N1', 'N2', 'N3', 'N6', 'N7', 'G10', 'G11', 'G16', 'G19', 'G22', 'G23']


**The ML Inference Bridge**

In [29]:
import pandas as pd
import torch

def predict_gate_delay(gate, model, scalers, extractor):
    # 1. Create the raw dictionary
    # We add 0.0 for the delay columns just to satisfy the adapt_dataset function
    raw_dict = {
        "NMOS_Width_nm": gate.wn_um * 1000,
        "PMOS_Width_nm": gate.wp_um * 1000,
        "Channel_Length_nm": 45.0,
        "Input_Slew_ps": gate.output_slew,
        "Load_Capacitance_fF": gate.cload_fF,
        "VDD_V": gate.vdd,
        "Temperature_C": gate.temp,
        "Threshold_Voltage_N_V": gate.vth_n,
        "Threshold_Voltage_P_V": gate.vth_p,
        "Gate_Type": gate.gate_type,
        "Process_Corner": gate.corner,
        # Dummy targets to prevent KeyError
        "rise_delay_ns": 0.0, 
        "fall_delay_ns": 0.0,
        "average_delay_ns": 0.0
    }
    
    # 2. Convert to DataFrame and Adapt
    df_single = pd.DataFrame([raw_dict])
    df_adapted = adapt_dataset(df_single)
    
    # 3. Handle One-Hot Encoding manually for the model
    # We need to make sure Gate_Type_INV, Gate_Type_NAND2, etc. all exist
    all_categories = {
        'Gate_Type': ['INV', 'NAND2', 'NOR2'],
        'Process_Corner': ['FF', 'FS', 'SF', 'SS', 'TT']
    }
    
    for feat, cats in all_categories.items():
        current_val = raw_dict[feat]
        for cat in cats:
            col_name = f"{feat}_{cat}"
            df_adapted[col_name] = 1.0 if cat == current_val else 0.0

    # 4. Extract Physics Features
    df_feat = extractor.extract(df_adapted)
    
    # 5. Final Scaling and Prediction
    # Ensure we only take the columns the model was trained on
    x_raw = scalers['raw'].transform(df_feat[RAW_FEATS].values.astype('float32'))
    x_phy = scalers['phy'].transform(df_feat[PHY_FEATS].values.astype('float32'))
    
    model.eval() # Ensure model is in eval mode
    with torch.no_grad():
        t_raw = torch.tensor(x_raw).float().to(CFG.DEVICE)
        t_phy = torch.tensor(x_phy).float().to(CFG.DEVICE)
        pred_log = model(t_raw, t_phy).cpu().numpy()
        
    # 6. Inverse transform and return average delay in PICOSECONDS
    delays_ns = scalers['y'].inverse_transform(pred_log)
    avg_delay_ns = delays_ns[0][2]
    
    return avg_delay_ns * 1000 # Convert ns to ps

**Run the Full Propagation**

In [30]:
import pandas as pd
import torch
import numpy as np

def run_static_timing_analysis(circuit, model, scalers, extractor):
    # Added limitation note for paper reviewer transparency
    #print("Note: Input slew propagation is approximated as a constant 15ps; iterative slew-aware STA is planned as future work.")
    
    all_gates = list(circuit.gates.values())
    
    # 1. Build ONE giant batch for the entire circuit
    raw_data = [{
        "Wn_um": g.wn_um, "Wp_um": g.wp_um, "L_um": 0.045, "VDD_V": g.vdd, 
        "TIN_ns": 0.015, # Assume stable 15ps input slew for vectorization
        "CLOAD_fF": g.cload_fF, "Temperature": g.temp,
        "Vth_N_V": g.vth_n, "Vth_P_V": g.vth_p,
        "N_series": 1.0 if g.gate_type=="INV" else 2.0,
        "Gate_Type_INV": 1.0 if g.gate_type=="INV" else 0.0,
        "Gate_Type_NAND2": 1.0 if g.gate_type=="NAND2" else 0.0,
        "Gate_Type_NOR2": 1.0 if g.gate_type=="NOR2" else 0.0,
        "Process_Corner_FF": 1.0 if g.corner=="FF" else 0.0,
        "Process_Corner_FS": 1.0 if g.corner=="FS" else 0.0,
        "Process_Corner_SF": 1.0 if g.corner=="SF" else 0.0,
        "Process_Corner_SS": 1.0 if g.corner=="SS" else 0.0,
        "Process_Corner_TT": 1.0 if g.corner=="TT" else 0.0,
    } for g in all_gates]
    
    # 2. Extract & Predict instantly (Zero Python loops!)
    df_batch = pd.DataFrame(raw_data)
    df_batch["Fanout"] = [len(g.fanout) if g.fanout else 4.0 for g in all_gates]
    df_feat = extractor.extract(df_batch)
    
    x_raw = scalers['raw'].transform(df_feat[RAW_FEATS].values.astype(np.float32))
    x_phy = scalers['phy'].transform(df_feat[PHY_FEATS].values.astype(np.float32))
    
    model.eval()
    with torch.no_grad():
        t_raw = torch.tensor(x_raw, device=CFG.DEVICE)
        t_phy = torch.tensor(x_phy, device=CFG.DEVICE)
        pred_log = model(t_raw, t_phy)
        
    delays_ns = scalers['y'].inverse_transform(pred_log.cpu().numpy())
    
    for i, gate in enumerate(all_gates):
        gate.delay = delays_ns[i][2] * 1000.0 # Convert to ps
        
    # 3. Lightning-fast topological propagation (Simple Addition)
    order = get_topological_order(circuit)
    for gate in order:
        if not gate.fanin:
            gate.arrival_time = 0.0
            gate.critical_input = None
        else:
            latest_input = max(gate.fanin, key=lambda x: x.arrival_time)
            gate.critical_input = latest_input
            gate.arrival_time = latest_input.arrival_time + gate.delay

**Critical Path Extraction (The Backtrace)**

In [31]:
def get_critical_path_report(circuit):
    # 1. Find the primary output gate with the highest arrival time
    outputs = [g for g in circuit.gates.values() if not g.fanout]
    end_gate = max(outputs, key=lambda x: x.arrival_time)
    
    # 2. Backtrace from the end to the beginning
    path = []
    current = end_gate
    while current is not None:
        path.append(current)
        current = current.critical_input # Jump to the gate that caused the delay
        
    path.reverse() # Flip it so it reads Input -> Output
    
    # 3. Print the Path Report
    print("\n" + "="*40)
    print("        CRITICAL PATH REPORT")
    print("="*40)
    for i, gate in enumerate(path):
        arrow = " -> " if i < len(path)-1 else ""
        print(f"{gate.name} ({gate.gate_type}) [Delay: {gate.delay:.2f}ps, Arrival: {gate.arrival_time:.2f}ps]{arrow}")
    
    print("="*40)
    print(f"TOTAL CIRCUIT DELAY: {path[-1].arrival_time:.2f} ps")

In [32]:
# 1. Run the analysis
run_static_timing_analysis(c17, ftnet, scalers, extractor) 

# 2. Extract and print the path
get_critical_path_report(c17)

Note: Input slew propagation is approximated as a constant 15ps; iterative slew-aware STA is planned as future work.

        CRITICAL PATH REPORT
N3 (INV) [Delay: 20.92ps, Arrival: 0.00ps] -> 
G11 (NAND2) [Delay: 15.91ps, Arrival: 15.91ps] -> 
G16 (NAND2) [Delay: 9.17ps, Arrival: 25.08ps] -> 
G22 (NAND2) [Delay: 9.17ps, Arrival: 34.25ps] -> 
G23 (NAND2) [Delay: 28.55ps, Arrival: 62.80ps]
TOTAL CIRCUIT DELAY: 62.80 ps


In [33]:
# 1. Create a folder for the benchmarks
!mkdir -p benchmarks

# 2. Download C17 and C432 directly from a reliable public source
# We use the raw file links so we don't need to clone a whole repo
!wget -O benchmarks/c17.blif https://raw.githubusercontent.com/cad-polito-it/testability/master/iscas85/c17.blif
!wget -O benchmarks/c432.blif https://raw.githubusercontent.com/cad-polito-it/testability/master/iscas85/c432.blif

# 3. Verify they are downloaded
import os
print("Downloaded files:", os.listdir("benchmarks"))

--2026-06-14 06:27:38--  https://raw.githubusercontent.com/cad-polito-it/testability/master/iscas85/c17.blif
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-06-14 06:27:39 ERROR 404: Not Found.

--2026-06-14 06:27:39--  https://raw.githubusercontent.com/cad-polito-it/testability/master/iscas85/c432.blif
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-06-14 06:27:39 ERROR 404: Not Found.

Downloaded files: ['c432.blif', 'c17.blif']


In [34]:
import re

def parse_blif(file_path, circuit):
    print(f"Parsing {file_path}...")
    
    with open(file_path, 'r') as f:
        raw_content = f.read()

    # Handle line continuations (the backslash \ at end of line)
    content = raw_content.replace('\\\n', ' ').splitlines()

    for line in content:
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('.model') or line.startswith('.end') or line.startswith('.outputs'):
            continue

        # 1. Handle Primary Inputs
        if line.startswith('.inputs'):
            inputs = line.split()[1:]
            for inp in inputs:
                circuit.add_gate(Gate(inp, "INV", wn_um=0.12, wp_um=0.24))

        # 2. Handle Logic (.names)
        elif line.startswith('.names'):
            parts = line.split()
            input_names = parts[1:-1]
            output_name = parts[-1]
            # Map based on input count so ML model doesn't crash
            g_type = "INV" if len(input_names) <= 1 else ("NAND2" if len(input_names) == 2 else "NOR2")
            new_gate = Gate(output_name, g_type, wn_um=0.12, wp_um=0.24)
            circuit.add_gate(new_gate)
            new_gate._temp_inputs = input_names

        # 3. Handle Gates (.gate)
        elif line.startswith('.gate') or line.startswith('.subckt'):
            parts = line.split()
            raw_g_type = parts[1].upper()
            
            # Normalize gate types for our ML model
            if "NAND" in raw_g_type:
                g_type = "NAND2"
            elif "NOR" in raw_g_type:
                g_type = "NOR2"
            else:
                g_type = "INV" # Default fallback
                
            mapping = {}
            in_names = []
            for p in parts[2:]:
                if '=' in p:
                    key, val = p.split('=')
                    if key.upper() in ['Y', 'OUT', 'Z', 'O']: mapping['out'] = val
                    else: in_names.append(val)
                else:
                    in_names.append(p)
                    
            out_name = mapping.get('out') or (in_names.pop() if in_names else None)
            
            if out_name:
                new_gate = Gate(out_name, g_type, wn_um=0.12, wp_um=0.24)
                circuit.add_gate(new_gate)
                new_gate._temp_inputs = in_names

    # Connect the parsed gates
    for name, gate in list(circuit.gates.items()):
        if hasattr(gate, '_temp_inputs'):
            for inp_name in gate._temp_inputs:
                if inp_name in circuit.gates:
                    circuit.connect(inp_name, name)
                else:
                    # Create missing primary inputs on the fly
                    pi = Gate(inp_name, "INV", 0.12, 0.24)
                    circuit.add_gate(pi)
                    circuit.connect(inp_name, name)
                    
    print(f"Done! Created a circuit with {len(circuit.gates)} nodes.")

In [35]:
c17_content = """
.model c17
.inputs 1 2 3 6 7
.outputs 22 23
.names 1 3 10
00 1
01 1
10 1
.names 3 6 11
00 1
01 1
10 1
.names 2 11 16
00 1
01 1
10 1
.names 11 7 19
00 1
01 1
10 1
.names 10 16 22
00 1
01 1
10 1
.names 19 22 23
00 1
01 1
10 1
.end
"""

with open("c17_manual.blif", "w") as f:
    f.write(c17_content.strip())

print("Successfully created c17_manual.blif locally!")

Successfully created c17_manual.blif locally!


In [36]:
# 1. Create a fresh circuit
c17_file = Circuit()

# 2. Parse the new manual file
parse_blif("c17_manual.blif", c17_file)

# 3. Calculate loads and run STA
c17_file.calculate_all_cloads()
run_static_timing_analysis(c17_file, ftnet, scalers, extractor)

# 4. Show the Path Report
get_critical_path_report(c17_file)

Parsing c17_manual.blif...
Done! Created a circuit with 11 nodes.
Note: Input slew propagation is approximated as a constant 15ps; iterative slew-aware STA is planned as future work.

        CRITICAL PATH REPORT
3 (INV) [Delay: 20.92ps, Arrival: 0.00ps] -> 
11 (NAND2) [Delay: 15.91ps, Arrival: 15.91ps] -> 
16 (NAND2) [Delay: 9.17ps, Arrival: 25.08ps] -> 
22 (NAND2) [Delay: 9.17ps, Arrival: 34.25ps] -> 
23 (NAND2) [Delay: 28.55ps, Arrival: 62.80ps]
TOTAL CIRCUIT DELAY: 62.80 ps


In [37]:
# Create a proper SERIAL chain where delays add up
logic_chain = [".model chain_test", ".inputs in_0", ".outputs out_100"]

# Gate 1 connects to in_0
logic_chain.append(".names in_0 out_1")
logic_chain.append("0 1")

# Gates 2 to 100 connect to the previous gate
for i in range(1, 100):
    logic_chain.append(f".names out_{i} out_{i+1}")
    logic_chain.append("0 1")

logic_chain.append(".end")

with open("real_chain_100.blif", "w") as f:
    f.write("\n".join(logic_chain))

print("Created a proper 100-gate serial chain!")

Created a proper 100-gate serial chain!


In [38]:
chain_circuit = Circuit()
parse_blif("real_chain_100.blif", chain_circuit)
chain_circuit.calculate_all_cloads()
run_static_timing_analysis(chain_circuit, ftnet, scalers, extractor)
get_critical_path_report(chain_circuit)

Parsing real_chain_100.blif...
Done! Created a circuit with 101 nodes.
Note: Input slew propagation is approximated as a constant 15ps; iterative slew-aware STA is planned as future work.

        CRITICAL PATH REPORT
in_0 (INV) [Delay: 8.98ps, Arrival: 0.00ps] -> 
out_1 (INV) [Delay: 8.98ps, Arrival: 8.98ps] -> 
out_2 (INV) [Delay: 8.98ps, Arrival: 17.95ps] -> 
out_3 (INV) [Delay: 8.98ps, Arrival: 26.93ps] -> 
out_4 (INV) [Delay: 8.98ps, Arrival: 35.90ps] -> 
out_5 (INV) [Delay: 8.98ps, Arrival: 44.88ps] -> 
out_6 (INV) [Delay: 8.98ps, Arrival: 53.86ps] -> 
out_7 (INV) [Delay: 8.98ps, Arrival: 62.83ps] -> 
out_8 (INV) [Delay: 8.98ps, Arrival: 71.81ps] -> 
out_9 (INV) [Delay: 8.98ps, Arrival: 80.78ps] -> 
out_10 (INV) [Delay: 8.98ps, Arrival: 89.76ps] -> 
out_11 (INV) [Delay: 8.98ps, Arrival: 98.74ps] -> 
out_12 (INV) [Delay: 8.98ps, Arrival: 107.71ps] -> 
out_13 (INV) [Delay: 8.98ps, Arrival: 116.69ps] -> 
out_14 (INV) [Delay: 8.98ps, Arrival: 125.66ps] -> 
out_15 (INV) [Delay: 8.98ps

In [39]:
# 1. Generate a 216-bit Ripple-Carry Adder locally (432 gates)
lines = [".model ripple_carry_adder_216bit"]

# Inputs: a0..a215, b0..b215, cin (433 inputs)
inputs = ["cin"] + [f"a{i}" for i in range(216)] + [f"b{i}" for i in range(216)]
lines.append(".inputs " + " ".join(inputs))

# Outputs: s0..s215, cout (217 outputs)
outputs = [f"s{i}" for i in range(216)] + ["cout"]
lines.append(".outputs " + " ".join(outputs))

current_carry = "cin"

for i in range(216):
    # SUM gate (3 inputs -> maps to NOR2 in our ML model)
    lines.append(f".names a{i} b{i} {current_carry} s{i}")
    lines.append("100 1\n010 1\n001 1\n111 1")

    # CARRY gate (3 inputs -> maps to NOR2 in our ML model)
    next_carry = f"c{i+1}" if i < 215 else "cout"
    lines.append(f".names a{i} b{i} {current_carry} {next_carry}")
    lines.append("11- 1\n1-1 1\n-11 1")
    
    current_carry = next_carry

lines.append(".end")

# Write to file
with open("rca_216bit.blif", "w") as f:
    f.write("\n".join(lines))

print("Successfully generated 'rca_216bit.blif' (216-bit Ripple-Carry Adder, 432 gates)!")

# 2. Run the STA Engine on our new solid benchmark
rca_circuit = Circuit()
parse_blif("rca_216bit.blif", rca_circuit)
rca_circuit.calculate_all_cloads()
run_static_timing_analysis(rca_circuit, ftnet, scalers, extractor)
get_critical_path_report(rca_circuit)

Successfully generated 'rca_216bit.blif' (216-bit Ripple-Carry Adder, 432 gates)!
Parsing rca_216bit.blif...
Done! Created a circuit with 865 nodes.
Note: Input slew propagation is approximated as a constant 15ps; iterative slew-aware STA is planned as future work.

        CRITICAL PATH REPORT
a0 (INV) [Delay: 21.59ps, Arrival: 0.00ps] -> 
c1 (NOR2) [Delay: 20.40ps, Arrival: 20.40ps] -> 
c2 (NOR2) [Delay: 20.40ps, Arrival: 40.81ps] -> 
c3 (NOR2) [Delay: 20.40ps, Arrival: 61.21ps] -> 
c4 (NOR2) [Delay: 20.40ps, Arrival: 81.61ps] -> 
c5 (NOR2) [Delay: 20.40ps, Arrival: 102.02ps] -> 
c6 (NOR2) [Delay: 20.40ps, Arrival: 122.42ps] -> 
c7 (NOR2) [Delay: 20.40ps, Arrival: 142.82ps] -> 
c8 (NOR2) [Delay: 20.40ps, Arrival: 163.23ps] -> 
c9 (NOR2) [Delay: 20.40ps, Arrival: 183.63ps] -> 
c10 (NOR2) [Delay: 20.40ps, Arrival: 204.03ps] -> 
c11 (NOR2) [Delay: 20.40ps, Arrival: 224.44ps] -> 
c12 (NOR2) [Delay: 20.40ps, Arrival: 244.84ps] -> 
c13 (NOR2) [Delay: 20.40ps, Arrival: 265.24ps] -> 
c14 (NO

In [43]:
import time, subprocess, re, os

print("="*50)
print("   FINAL PAPER METRICS: ACCURACY & SPEEDUP")
print("="*50)

# ── 1. ML-STA runtime for C17 ─────────────────────────
t0 = time.time()
run_static_timing_analysis(c17, ftnet, scalers, extractor)
ml_sta_time_c17 = time.time() - t0
ml_delay_c17 = max(
    [g for g in c17.gates.values() if not g.fanout],
    key=lambda x: x.arrival_time
).arrival_time

# ── 2. ML INFERENCE time only for 432-gate circuit ────
# Reuse the same batch-build logic already inside run_static_timing_analysis
# but time ONLY the forward pass, not the graph traversal

all_gates_432 = list(rca_circuit.gates.values())

# Build the DataFrame batch exactly as run_static_timing_analysis does
raw_data = [{
    "Wn_um": g.wn_um, "Wp_um": g.wp_um, "L_um": 0.045, "VDD_V": g.vdd,
    "TIN_ns": 0.015,
    "CLOAD_fF": g.cload_fF, "Temperature": g.temp,
    "Vth_N_V": g.vth_n, "Vth_P_V": g.vth_p,
    "N_series": 1.0 if g.gate_type == "INV" else 2.0,
    "Gate_Type_INV":   1.0 if g.gate_type == "INV"   else 0.0,
    "Gate_Type_NAND2": 1.0 if g.gate_type == "NAND2" else 0.0,
    "Gate_Type_NOR2":  1.0 if g.gate_type == "NOR2"  else 0.0,
    "Process_Corner_FF": 1.0 if g.corner == "FF" else 0.0,
    "Process_Corner_FS": 1.0 if g.corner == "FS" else 0.0,
    "Process_Corner_SF": 1.0 if g.corner == "SF" else 0.0,
    "Process_Corner_SS": 1.0 if g.corner == "SS" else 0.0,
    "Process_Corner_TT": 1.0 if g.corner == "TT" else 0.0,
} for g in all_gates_432]

df_batch = pd.DataFrame(raw_data)
df_batch["Fanout"] = [len(g.fanout) if g.fanout else 4.0 for g in all_gates_432]
df_feat  = extractor.extract(df_batch)

x_raw = scalers['raw'].transform(df_feat[RAW_FEATS].values.astype(np.float32))
x_phy = scalers['phy'].transform(df_feat[PHY_FEATS].values.astype(np.float32))

t_raw = torch.tensor(x_raw, device=CFG.DEVICE)
t_phy = torch.tensor(x_phy, device=CFG.DEVICE)

# Time ONLY the forward pass
ftnet.eval()
with torch.no_grad():
    t0 = time.time()
    _ = ftnet(t_raw, t_phy)
    ml_inference_time_c432 = time.time() - t0

# ── 3. SPICE C17 — level=1 (same physics as training) ─
os.system("apt-get install ngspice -y -qq > /dev/null 2>&1")
print("Running ngspice C17 simulation...")

spice_c17 = """* C17 ISCAS85 45nm TT corner
.model nmos nmos level=1 vto=0.32 kp=180u lambda=0.01
.model pmos pmos level=1 vto=-0.30 kp=90u  lambda=0.01
Vdd vdd 0 DC 1.0
VN1 N1 0 PWL(0 1.0  50p 1.0  65p 0.0)
VN3 N3 0 DC 1.0
VN6 N6 0 DC 1.0
VN2 N2 0 DC 0.0
VN7 N7 0 DC 0.0
MG10p1 G10 N1  vdd vdd pmos L=45n W=240n
MG10p2 G10 N3  vdd vdd pmos L=45n W=240n
MG10n1 G10 N1  mid10 0 nmos L=45n W=120n
MG10n2 mid10 N3 0   0 nmos L=45n W=120n
MG11p1 G11 N3  vdd vdd pmos L=45n W=240n
MG11p2 G11 N6  vdd vdd pmos L=45n W=240n
MG11n1 G11 N3  mid11 0 nmos L=45n W=120n
MG11n2 mid11 N6 0   0 nmos L=45n W=120n
MG16p1 G16 N2  vdd vdd pmos L=45n W=240n
MG16p2 G16 G11 vdd vdd pmos L=45n W=240n
MG16n1 G16 N2  mid16 0 nmos L=45n W=120n
MG16n2 mid16 G11 0  0 nmos L=45n W=120n
MG19p1 G19 G11 vdd vdd pmos L=45n W=240n
MG19p2 G19 N7  vdd vdd pmos L=45n W=240n
MG19n1 G19 G11 mid19 0 nmos L=45n W=120n
MG19n2 mid19 N7 0   0 nmos L=45n W=120n
MG22p1 G22 G10 vdd vdd pmos L=45n W=240n
MG22p2 G22 G16 vdd vdd pmos L=45n W=240n
MG22n1 G22 G10 mid22 0 nmos L=45n W=120n
MG22n2 mid22 G16 0  0 nmos L=45n W=120n
MG23p1 G23 G19 vdd vdd pmos L=45n W=240n
MG23p2 G23 G22 vdd vdd pmos L=45n W=240n
MG23n1 G23 G19 mid23 0 nmos L=45n W=120n
MG23n2 mid23 G22 0  0 nmos L=45n W=120n
Cout G23 0 15f
.option temp=25
.tran 0.5p 1n
.measure tran tpd_c17 TRIG v(N1) VAL=0.5 FALL=1 TARG v(G23) VAL=0.5 RISE=1
.end"""

with open("c17_spice.sp","w") as f: f.write(spice_c17)

t0 = time.time()
subprocess.run("ngspice -b c17_spice.sp -o c17.log",
               shell=True, capture_output=True)
spice_time_c17 = time.time() - t0

try:
    txt = open("c17.log").read()
    m = re.search(r"tpd_c17\s*=\s*([0-9eE.+-]+)", txt)
    spice_delay_c17 = float(m.group(1)) * 1e12 if m else None
    if spice_delay_c17 is None:
        print("MEASURE FAILED — log snippet:")
        print(txt[-800:])   # print end of log to diagnose
except Exception as e:
    spice_delay_c17 = None
    print(f"Log read failed: {e}")

# ── 4. SPICE scaling proxy — 50-gate chain, level=1 ───
chain = "* 50-INV chain\n"
chain += ".model nmos nmos level=1 vto=0.32 kp=180u lambda=0.01\n"
chain += ".model pmos pmos level=1 vto=-0.30 kp=90u  lambda=0.01\n"
chain += "Vdd vdd 0 1.0\nVin n0 0 PWL(0 0 50p 0 65p 1.0)\n"
for k in range(50):
    inp = f"n{k}"; out = f"n{k+1}"
    chain += f"Mn{k} {out} {inp} 0   0 nmos L=45n W=120n\n"
    chain += f"Mp{k} {out} {inp} vdd vdd pmos L=45n W=240n\n"
    chain += f"Cl{k} {out} 0 10f\n"
chain += ".option temp=25\n.tran 2p 10n\n.end\n"

with open("chain50.sp","w") as f: f.write(chain)
t0 = time.time()
subprocess.run("ngspice -b chain50.sp -o chain50.log",
               shell=True, capture_output=True)
spice_time_50 = time.time() - t0

spice_time_432_est = spice_time_50 * (432/50)
speedup_c432 = spice_time_432_est / ml_inference_time_c432

# ── 5. Print results ───────────────────────────────────
if spice_delay_c17:
    err = abs(ml_delay_c17 - spice_delay_c17) / spice_delay_c17 * 100
    print(f"\n[C17 Benchmark — 6 Gates]")
    print(f"  ML-STA Delay   : {ml_delay_c17:.2f} ps")
    print(f"  SPICE Delay    : {spice_delay_c17:.2f} ps  (ngspice, same 45nm params as training)")
    print(f"  Accuracy Error : {err:.2f}%")
    print(f"  ML-STA Runtime : {ml_sta_time_c17*1000:.1f} ms")
    print(f"  SPICE Runtime  : {spice_time_c17*1000:.1f} ms")
    print(f"  Speedup        : {spice_time_c17/ml_sta_time_c17:.0f}x")
else:
    print("C17 SPICE measurement failed — check log")

print(f"\n[Scaling: 50-gate chain → 432-gate estimate]")
print(f"  50-gate SPICE runtime : {spice_time_50*1000:.1f} ms")
print(f"  Estimated 432-gate    : {spice_time_432_est*1000:.0f} ms  (linear extrapolation)")
print(f"  ML inference 432-gate : {ml_inference_time_c432*1000:.1f} ms  (vectorised forward pass)")
print(f"  Estimated speedup     : {speedup_c432:.0f}x")
print("="*50)

   FINAL PAPER METRICS: ACCURACY & SPEEDUP
Note: Input slew propagation is approximated as a constant 15ps; iterative slew-aware STA is planned as future work.
Running ngspice C17 simulation...

[C17 Benchmark — 6 Gates]
  ML-STA Delay   : 62.80 ps
  SPICE Delay    : 63.32 ps  (ngspice, same 45nm params as training)
  Accuracy Error : 0.82%
  ML-STA Runtime : 12.6 ms
  SPICE Runtime  : 64.7 ms
  Speedup        : 5x

[Scaling: 50-gate chain → 432-gate estimate]
  50-gate SPICE runtime : 11.1 ms
  Estimated 432-gate    : 96 ms  (linear extrapolation)
  ML inference 432-gate : 1.9 ms  (vectorised forward pass)
  Estimated speedup     : 49x
